# React — Advanced forms

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> Topic 10 gave you controlled inputs, one state object, `onSubmit` and a pure `validate()`.
> This topic takes that as far as it goes in plain React — repeated fields and validation as
> data — and then introduces **Actions**, React 19's built-in way to run a submission.
>
> **A note on Server Functions.** React's Actions also work with functions marked `'use server'`,
> which run on a server and enable forms that work before JavaScript loads. That needs a
> server-capable framework, which this course does not use. Everything here is **client-side
> Actions**, which is the same API without that half.

## LESSON 78 — Field arrays and dependent fields

Two shapes that break a naive form, and both are really about state structure rather than about
forms.

### Repeated fields

"Add another attendee". The form now has a variable number of inputs, and the state that backs
them is an **array of objects** — LESSON 28's drill, which is why that one was worth doing.

```jsx
const [rows, setRows] = useState([{ id: 1, name: "", email: "" }]);
```

Three operations, all immutable, and one detail that matters more than the rest:

```jsx
const addRow = () =>
  setRows([...rows, { id: nextId++, name: "", email: "" }]);

const removeRow = (id) =>
  setRows(rows.filter((row) => row.id !== id));

const updateRow = (id, field, value) =>
  setRows(rows.map((row) => (row.id === id ? { ...row, [field]: value } : row)));
```

**Every row needs a stable `id`, generated when the row is created.** Not the array index —
LESSON 20 showed you what index keys do when something is removed, and in a form the consequence
is worse than a wasted render: the user deletes row 2 and the text they typed in row 3 appears to
move up into it.

### Dependent fields

A field whose *options* or *validity* depend on another: country → region, "other" → a free-text
box, a start date that bounds an end date.

Two rules cover almost all of it.

**Do not copy what you can derive.** The list of regions for the chosen country is a
`const regions = REGIONS[values.country] ?? []` above the return — LESSON 29. Putting it in state
gives you two things to keep in step.

**Clear the dependent value when the thing it depends on changes.** This one *is* state, and it
is the bug people ship:

```jsx
function setCountry(country) {
  setValues({ ...values, country, region: "" });   // the old region is meaningless now
}
```

Leave it out and the form quietly submits `{ country: "Italy", region: "Bavaria" }`, having shown
the user a region select that no longer contains their selection.

### Key Notes

- Repeated fields are an array of objects in state: add with spread, remove with `filter`, update
  with `map`.
- Give each row an `id` at creation. Index keys move the user's typing between rows.
- Derive dependent *options*; never copy them into state.
- When a parent field changes, clear the dependent field in the same update.

### Example

**Runnable — plain JS.** The three array operations and the dependent-field reset, as pure
functions. This is exactly the code that goes in your component — the only thing missing is the
`setState` around it.

In [ ]:
// L78 — rows, and a dependent field

let l78NextId = 3;

const l78Rows = [
  { id: 1, name: "Ada", email: "ada@example.com" },
  { id: 2, name: "Grace", email: "" },
];

const l78Add = (rows) => [...rows, { id: (l78NextId += 1), name: "", email: "" }];
const l78Remove = (rows, id) => rows.filter((row) => row.id !== id);
const l78Update = (rows, id, field, value) =>
  rows.map((row) => (row.id === id ? { ...row, [field]: value } : row));

const l78Added = l78Add(l78Rows);
console.log("after add:   ", l78Added.map((r) => `${r.id}:${r.name || "(blank)"}`).join(", "));

const l78Edited = l78Update(l78Added, 2, "email", "grace@example.com");
console.log("after edit:  ", l78Edited.find((r) => r.id === 2));

const l78Removed = l78Remove(l78Edited, 1);
console.log("after remove:", l78Removed.map((r) => r.id));
console.log("original untouched:", l78Rows.length === 2);

// --- the dependent field ----------------------------------------------------
const l78Regions = {
  Italy: ["Lazio", "Lombardia", "Sicilia"],
  Germany: ["Bayern", "Hessen"],
};

// derived, never stored
const l78OptionsFor = (country) => l78Regions[country] ?? [];

function l78SetCountry(values, country) {
  return { ...values, country, region: "" };      // the reset that people forget
}

let l78Values = { country: "Italy", region: "Sicilia" };
console.log("\noptions:", l78OptionsFor(l78Values.country));

const l78Careless = { ...l78Values, country: "Germany" };
const l78Correct = l78SetCountry(l78Values, "Germany");

console.log("careless:", l78Careless, "— region valid?", l78OptionsFor("Germany").includes(l78Careless.region));
console.log("correct :", l78Correct, "— region valid?", l78Correct.region === "");

### Exercise

**Runnable — plain JS.**

1. Add `l78Move(rows, id, direction)` that moves a row up or down without mutating the array, and
   returns the array unchanged if the move is impossible. Show it working and show the original
   array untouched.
2. Write `l78Validate(rows)` returning an array of `{ id, errors }` — a row needs a name, and an
   email containing `@` — plus a form-level error when two rows share an email. Run it on a set
   with one blank name and one duplicated email.
3. Write `l78Reconcile(values, changed)` that takes the whole values object and the name of the
   field that just changed, and clears every field that depends on it. Support two dependencies:
   `region` depends on `country`, and `city` depends on `region`. Change the country and show that
   **both** are cleared.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** The index-key bug, made concrete without React.

Model a row's identity twice: once with `id` and once with the array index. Write
`l78TrackByIndex(rows, removedIndex)` and `l78TrackById(rows, removedId)` that both remove a row
and then report, for each remaining row, *what key it now has* and *what data it holds*.

Show the case that breaks: three rows, the user has typed in all three, row 2 is removed. Print
which key now points at which text under each scheme.

Then answer in a comment: React reuses the DOM node for an unchanged key. Using that sentence,
say exactly what the user sees in the index version — and why this is worse in a form than in a
read-only list.

In [ ]:
// Your code here

## LESSON 79 — Validation as data

LESSON 33 gave you `validate(values) → errors`, a pure function, and that is still the right
shape. This lesson is about what goes *inside* it once a form has fifteen fields, and about
**when** the errors are allowed to appear.

### Rules as data

Written as code, a validator grows one `if` per rule and becomes a wall:

```js
if (!values.email) errors.email = "Email is required";
else if (!values.email.includes("@")) errors.email = "That doesn't look like an email";
if (!values.age) errors.age = "Age is required";
else if (Number(values.age) < 18) errors.age = "You must be 18 or over";
// …thirty more
```

Written as data, the rules become a list you can read, extend and reuse:

```js
const rules = {
  email: [required("Email"), matches(/@/, "That doesn't look like an email")],
  age: [required("Age"), min(18, "You must be 18 or over")],
};
```

`required`, `min` and `matches` are tiny functions returning a message or `null`. The runner is
about eight lines. What you gain is not cleverness — it is that the rules are now a *value*: you
can define them next to the form, share them between the add and edit forms, test them without a
component (topic 26), and add a rule without touching the runner.

The example cell builds the whole thing, and it is genuinely reusable code.

**Do not over-build it.** A three-field form does not need a rule engine; `if` statements are
clearer. The threshold is roughly when the same rules appear in two places or the validator stops
fitting on a screen.

### When errors should appear

Correct rules shown at the wrong moment feel hostile. The convention that works:

| moment | show errors for |
|---|---|
| while typing in a field the user has not left | **nothing** |
| on blur (they left the field) | that field |
| on submit | every field, and focus the first one |
| while typing in a field that already shows an error | that field, live — so they see it clear |

The mechanism is one extra piece of state: which fields have been **touched**. Errors exist for
every field from the first render; `touched` decides which of them are visible.

```js
const visible = Object.fromEntries(
  Object.entries(errors).filter(([field]) => touched[field] || submitted),
);
```

That is again a derived value (LESSON 29), not a second copy of the errors.

### Async checks

"Is this username taken?" cannot be answered by a pure function. Three rules keep it sane:

1. **Debounce it** (LESSON 42) — do not ask on every keystroke.
2. **Keep it separate from the synchronous errors.** A pending check is not an error; it is a
   third state, and the submit button should know about it.
3. **Check again on submit.** Anything decided asynchronously can be stale by the time they press
   the button, and the server is the only authority.

### And when a form library is worth it

Honestly: this course teaches none, and everything above is a few dozen lines. A library
(React Hook Form, Formik, TanStack Form) starts to earn its place when you have many large forms
sharing rules, when you need per-field subscriptions for performance on a very large form, or when
you want schema validation (Zod, Yup) shared with your backend types.

What you should not do is reach for one before you have written this by hand once. Every library
above is an opinionated arrangement of exactly these pieces — values, errors, touched, submit
state — and they are much easier to evaluate once you have built them.

### Key Notes

- Keep `validate(values) → errors` pure; express the rules as **data** once there are enough of
  them to repeat or share.
- Errors are computed always, and *shown* according to `touched` + `submitted`.
- Async checks are debounced, tracked separately from errors, and repeated on submit.
- A library is worth it for many large forms or shared schemas — not for your first one.

### Example

**Runnable — plain JS.** A rules-as-data validator, the visibility rule, and both together. Real
code for your project.

In [ ]:
// L79 — rules as data

const l79Required = (label) => (value) =>
  value === undefined || value === null || String(value).trim() === "" ? `${label} is required` : null;

const l79Min = (limit, message) => (value) => (Number(value) < limit ? message : null);

const l79Matches = (pattern, message) => (value) =>
  value && !pattern.test(String(value)) ? message : null;

const l79SameAs = (field, message) => (value, values) =>
  value !== values[field] ? message : null;

const l79Rules = {
  email: [l79Required("Email"), l79Matches(/@/, "That doesn't look like an email")],
  age: [l79Required("Age"), l79Min(18, "You must be 18 or over")],
  password: [l79Required("Password"), l79Matches(/.{8,}/, "At least 8 characters")],
  confirm: [l79Required("Confirmation"), l79SameAs("password", "The passwords do not match")],
};

// the whole runner
function l79Validate(values, rules = l79Rules) {
  const errors = {};
  for (const [field, checks] of Object.entries(rules)) {
    for (const check of checks) {
      const message = check(values[field], values);
      if (message) {
        errors[field] = message;      // first failing rule wins
        break;
      }
    }
  }
  return errors;
}

const l79Values = { email: "ada", age: "16", password: "secret", confirm: "secrets" };
console.log("errors:", l79Validate(l79Values));

const l79Good = { email: "ada@example.com", age: "42", password: "correcthorse", confirm: "correcthorse" };
console.log("valid form:", l79Validate(l79Good), "— empty means valid");

// --- which of them the user should currently SEE ----------------------------
function l79Visible(errors, touched, submitted) {
  return Object.fromEntries(
    Object.entries(errors).filter(([field]) => submitted || touched[field]),
  );
}

const l79Errors = l79Validate(l79Values);
console.log("\nstill typing, nothing touched:", l79Visible(l79Errors, {}, false));
console.log("left the email field         :", l79Visible(l79Errors, { email: true }, false));
console.log("pressed submit               :", Object.keys(l79Visible(l79Errors, {}, true)));

### Exercise

**Runnable — plain JS**, building on the example.

1. Add `l79OneOf(list, message)` and `l79MaxLength(n, message)`, and a `bio` field that is
   optional but at most 100 characters. Careful: "optional" means an empty value must **pass**.
   Prove it with an empty bio and a 120-character one.
2. Collect **all** failing rules per field instead of only the first, then explain in a comment
   why showing one message at a time is usually the better interface even though you now have all
   of them.
3. Write `l79FirstInvalid(errors, order)` returning the field name a submit handler should focus,
   given the fields' display order. Explain in one sentence why the order matters and why
   `Object.keys(errors)` is not good enough.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** The async check, with the two failures it usually ships with.

Write `l79MakeUsernameChecker(lookup)` returning `check(username)` that resolves to
`{ status: "free" | "taken" | "error" }`, plus a small state machine `l79Field` holding
`{ value, syncError, asyncStatus }` and a function `l79CanSubmit(field)`.

Then demonstrate two bugs and fix them:

1. **The stale answer.** Type `ada`, then `adam` before the first check returns, and have the
   `ada` response arrive last. Show the field ending up in the wrong state, then fix it with
   LESSON 43's technique.
2. **Submitting during a pending check.** Show `l79CanSubmit` returning `true` while a check is in
   flight, and decide what it should do instead — and say why "block the button" and "let them
   submit and check on the server" are both defensible.

In [ ]:
// Your code here

## LESSON 80 — Actions

Since LESSON 32 you have submitted forms like this:

```jsx
<form onSubmit={handleSubmit}>      // and handleSubmit calls e.preventDefault()
```

React 19 adds another way: pass a **function** to `action`.

```jsx
function search(formData) {
  const query = formData.get("query");
  // …
}

<form action={search}>
  <input name="query" />
  <button type="submit">Search</button>
</form>
```

No `event`, no `preventDefault`, and the function receives a `FormData` object built from the
fields' **`name` attributes**. That is why every input in this lesson has a `name`: it is the key
in the form data, exactly as in a plain HTML form.

### What React does for you

React's own comparison is worth reading closely:

> Reading form data with `onSubmit` works in every version of React and gives you direct access to
> the submit event… Passing the function to the `action` prop instead runs the submission in a
> Transition. React then tracks the pending state, sends thrown errors to the nearest error
> boundary, and lets the form work with `useActionState` and `useOptimistic`.

Four things, then. It is a **Transition** (LESSON 72), so the page stays responsive while it runs.
The **pending state is tracked** for you — LESSON 81 and 82 read it. Thrown **errors go to the
error boundary** (LESSON 75). And it unlocks the two Hooks in the next lessons.

Plus one behaviour to know precisely:

> After the `action` function succeeds, all uncontrolled field elements in the form are reset.

Experiment 34 measures exactly what "reset" means. Submitting a form with three fields:

| field | before submit | after |
|---|---|---|
| `title` — uncontrolled, `defaultValue=""` | "Release notes" | **""** |
| `tag` — uncontrolled, `defaultValue="react"` | "react-19" | **"react"** |
| `body` — controlled | "edited body" | "edited body" |

So "reset" means *back to the default value*, not "cleared", and controlled fields are untouched
because their value comes from state that React does not own.

The action may be `async`. React keeps the form pending until the returned Promise settles.

### The trap: throwing for an expected failure

Errors thrown in an action go to the error boundary. That is right for a genuine crash and wrong
for "the name is required" — and experiment 34 shows why. Tick "make the action throw" and
submit: the boundary's fallback replaces **the entire form**, along with everything the user had
typed.

So:

| the failure | how the action ends |
|---|---|
| validation, a rejected save, a 409 conflict | **return** a value describing it (LESSON 81) |
| something genuinely unexpected | throw, and let the boundary catch it |

### Which should you use?

`onSubmit` is not deprecated and is still the right choice when you need the event itself — to
read `event.target`, to stop propagation, or to do something before default handling. Actions are
the better default for an ordinary submit, because everything in the list above comes free and
LESSON 81's Hook builds directly on them.

### Key Notes

- `<form action={fn}>` calls `fn(formData)` — no event, no `preventDefault`; fields are read by
  their `name`.
- It runs in a Transition, tracks pending, and sends thrown errors to the error boundary.
- On success React resets **uncontrolled** fields to their defaults; controlled fields keep their
  state.
- Return expected failures; throw only for the unexpected, or the boundary takes the form away.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/34-form-actions.jsx`.

Type into all three fields and submit, then look at which ones changed — `window.__lastSubmission`
holds what the action received. Then tick the checkbox and submit again to watch a thrown error
take the whole form with it.

### Exercise

**In the playground**, in `34-form-actions.jsx`.

1. Add a `<select name="priority">` with three options and a checkbox `<input type="checkbox"
   name="urgent">`. Submit with the box ticked and then unticked, and report exactly what
   `FormData` contains each time. The unticked case surprises everyone — say what you would do
   about it in a real form.
2. Remove the `name` from the title input and submit. What arrives in the action? Explain in one
   sentence why, in terms of what `FormData` is built from.
3. Make the action `async` with a 3-second wait and submit. Is the form usable while it runs? Is
   the page? Connect what you see to the word "Transition" in React's description.
4. Replace the thrown error with a **returned** object `{ ok: false, message: "…" }` and display
   it. What breaks? (Nothing renders it yet — that is LESSON 81, and this is the problem it
   solves.)

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** `FormData` is a browser API and worth knowing on its own; Deno has it, so
this really runs.

Build a `FormData` by hand with `append`, then answer with code:

1. What does `get` return for a key that was never set, and how is that different from an empty
   string?
2. Append the same key twice. What does `get` return, and what does `getAll` return?
3. Convert it to a plain object with `Object.fromEntries`. Which of the two values from (2)
   survives, and what does that mean for a form with several checkboxes sharing a name?
4. Write `l80ToObject(formData)` that keeps repeated keys as arrays and single keys as values, and
   show it on a form with one `tags` key appearing three times.

In [ ]:
// Your code here

## LESSON 81 — `useActionState`

LESSON 80 ended with a problem: an action can *return* a result, and nothing renders it. This is
the Hook that does.

> `useActionState` is a React Hook that lets you update state with side effects using Actions.

```jsx
const [state, formAction, isPending] = useActionState(action, initialState);
```

Three things back, and one surprise going in.

### The action's signature changes

This is the detail everyone trips over, and React flags it:

> When using `useActionState` with a `<form>` action prop, the [action] receives an extra argument
> as its first argument: the previous or initial state. The submitted form data is therefore its
> second argument instead of its first.

```jsx
async function saveName(previousState, formData) {
  const name = formData.get("name").trim();
  if (!name) return { ok: false, message: "Name is required." };

  await api.save(name);
  return { ok: true, message: `Saved ${name}.` };
}
```

Compare it with LESSON 17's reducer and the shape is familiar: previous state in, next state out.
The difference is that this one may be `async` and may have side effects — which a reducer must
never have.

### What comes back

- **`state`** — the initial value on the first render, then whatever the action last returned.
- **`formAction`** — pass it to `<form action={formAction}>`.
- **`isPending`** — true while the action is running.

Measured in experiment 35, submitting twice:

| moment | `state.message` | `isPending` | `attempt` |
|---|---|---|---|
| first render | "not submitted yet" | false | 0 |
| submitting an empty name | "not submitted yet" | **true** | 0 |
| after | "Name is required." | false | 1 |
| submitting "Ada" | "Name is required." | **true** | 1 |
| after | "Saved Ada." | false | 2 |

Read the rows in bold: **while an action runs, the state is still the previous one.** There is no
in-between blank state, which is exactly what you want — the old message stays until a new one
replaces it. And the console shows the action receiving `previousState` each time, which is how
`attempt` counts up without any separate counter.

### Why this replaces four `useState`s

The version you would otherwise write:

```jsx
const [message, setMessage] = useState("");
const [ok, setOk] = useState(null);
const [pending, setPending] = useState(false);
const [error, setError] = useState(null);
```

Four pieces of state that must be updated together and can contradict each other — LESSON 67's
argument, in the place it does the most damage. `useActionState` gives you one state object and a
pending flag React maintains, so "pending and showing an error" or "ok and still loading" are not
representable.

### Returning, not throwing

The rule from LESSON 80, now with somewhere to put the result: an expected failure is a **return
value**. The form stays on screen, the fields keep their contents, and the message renders. In
experiment 35 the empty-name submission returns `{ ok: false }`, and the form is entirely intact
afterwards — unlike experiment 34's throw, which took the form with it.

One caveat: React resets uncontrolled fields **after a successful action**, and the action
returning `{ ok: false }` still counts as success as far as the form is concerned. If losing the
input on a validation failure matters, use controlled inputs for that form.

### Key Notes

- `const [state, formAction, isPending] = useActionState(action, initialState)`.
- The action is `(previousState, formData)` — the state comes **first**.
- Return the result; the returned value becomes the new state and renders.
- While pending, the previous state is still shown. One state object instead of four flags.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/35-action-state.jsx`.

Submit with the field empty, then with a name. Watch `state.message`, `state.attempt` and
`isPending`, and read the console line showing what the action received. `window.__actionCalls`
keeps the record.

### Exercise

**In the playground**, in `35-action-state.jsx`.

1. Swap the action's parameters to `(formData, previousState)` and submit. What error do you get,
   and why does it point at `formData.get`? Put them back.
2. Make the action return only a string instead of an object, and render it. Then say which shape
   you would ship for a form with per-field errors, and why (LESSON 67 again).
3. Use `state.attempt` to show "You have tried 3 times — need help?" after the third failure. This
   is the reason the action gets the previous state: do it **without** adding a `useState`.
4. Add a second form on the same page with its own `useActionState`. Confirm the two are
   independent, and say in one sentence what would have happened if you had shared one Hook
   between them (LESSON 64).

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** The action is a reducer that is allowed to be async, so model it as one.

Write `l81Run(action, initialState, submissions)` that feeds a list of form-data objects through
an async action one at a time, threading the returned state into the next call, and returns the
final state plus a log of every transition.

Use it with an action that: rejects an empty name, refuses a name already in a `saved` list, and
otherwise adds it. Feed it `["", "Ada", "Ada", "Grace"]` and print the log.

Then answer in comments: your `l81Run` awaits each submission before starting the next. React does
not queue submissions that way — what happens in a real form if the user presses submit twice
quickly, and which of the tools you have met (LESSON 82's pending flag, a disabled button, an
idempotent server) would you use, and why more than one?

In [ ]:
// Your code here

## LESSON 82 — `useFormStatus`

`useActionState` gives the pending flag to the component that owns the form. But the component
that usually *needs* it is the submit button — often a shared `<SubmitButton />` used by every
form in the app, which knows nothing about any particular action.

That is the gap this Hook fills.

> `useFormStatus` is a Hook that gives you status information of the last form submission.

```jsx
import { useFormStatus } from "react-dom";       // react-dom, not react

function SubmitButton() {
  const { pending } = useFormStatus();
  return (
    <button type="submit" disabled={pending}>
      {pending ? "Saving…" : "Save"}
    </button>
  );
}
```

`<SubmitButton />` now works inside **any** form, with no props, and every form gets consistent
pending behaviour. Note the import: it comes from **`react-dom`**, because it is about DOM forms.

### What it returns

| field | what it is |
|---|---|
| `pending` | `true` while the parent `<form>` is submitting |
| `data` | the `FormData` being submitted, or `null` |
| `method` | `'get'` or `'post'` |
| `action` | the function passed to the parent form's `action` prop, or `null` |

`data` is more useful than it looks: it lets the button say *what* is being saved —
`Saving "${data.get('title')}"…` — without the form passing anything down.

### The rule that makes it work, and breaks it

> The `useFormStatus` Hook must be called from a component that is rendered inside a `<form>`.
>
> `useFormStatus` will only return status information for a parent `<form>`. It will not return
> status information for any `<form>` rendered in that same component or children components.

So the Hook reads the form **above** it in the tree, not the one beside it. Experiment 35 calls it
in both places at once and measures the difference during a live submission:

| where it is called | `pending` while submitting |
|---|---|
| in `<SubmitButton />`, rendered inside the form | **true** |
| in the component that renders the `<form>` | **false** |

The second row is not a bug and not a race — it is the documented behaviour, and it is the single
most common `useFormStatus` question. If you need the pending state in the component that owns the
form, you already have it: `useActionState` returns it.

### Which pending flag to use

| you are in | use |
|---|---|
| the component that renders the form | `isPending` from `useActionState` |
| a component inside the form (a button, a field, a footer) | `useFormStatus` |

They are the same information from two directions, and neither replaces the other.

### Key Notes

- Import from **`react-dom`**. Returns `{ pending, data, method, action }`.
- It reads the nearest **parent** form — so it must be called in a child component.
- Called in the component that renders the form, `pending` is always `false`. Measured, and
  documented.
- Its point is a shared submit button that needs no props.

### Example

**In the playground**, in `35-action-state.jsx`, which already has both calls.

Submit and watch the two values on screen: the button says "saving…" and is disabled, while the
`useFormStatus` reading from the outer component still says `false`. `window.__childStatus` and
`window.__sameComponentStatus` hold both.

### Exercise

**In the playground**, in `35-action-state.jsx`.

1. Use `status.data` to make the button read `Saving "Ada"…` during the submission. What is `data`
   before any submission, and what must you do about that?
2. Add a `<Fieldset>` component inside the form that disables its children while pending, using
   the same Hook. Confirm it works without any prop being passed to it.
3. Move `<SubmitButton />` **outside** the `<form>` and submit. Does the button still work, and
   what does `pending` say? Explain using the rule about parent forms.
4. Add a second form on the page, each with its own submit button, and submit one. Does the other
   button go pending? Say what that tells you about how the Hook finds "the form".

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**In your project** — no cell, because the subject is a component you will actually keep.

Design the `<SubmitButton />` you would use in every form of Mini-project 4. Write it out, then
answer:

1. What props does it take, and why is `pending` **not** one of them?
2. It needs to say something different per form ("Save", "Send invitation"). Is that a `label`
   prop or `children`? Justify it with LESSON 67.
3. A form has two submit buttons — "Save" and "Save and add another". `useFormStatus` tells both
   that the form is pending, but not which one was pressed. How would you find out? (Look at
   `status.data` and at what a `<button name value>` contributes to `FormData`.)
4. What should the button do when the action fails — and which of the two Hooks in these lessons
   knows that it did?

## LESSON 83 — `useOptimistic`

The last piece of the Actions story. Everything so far has made waiting *visible*; this makes it
mostly invisible.

> `useOptimistic` is a React Hook that lets you optimistically update the UI.

```jsx
const [optimisticMessages, addOptimistic] = useOptimistic(
  messages,                                                  // the real state
  (current, newText) => [...current, { id: "temp", text: newText, sending: true }],
);
```

Two arguments: the real value, and a **reducer** that describes what the UI should look like while
something is in flight. It must be pure — LESSON 52's rule, again.

You then render `optimisticMessages` instead of `messages`, and call `addOptimistic(text)` at the
start of your Action.

### The three phases, measured

Experiment 36 sends a message against a 700 ms mock server:

| moment | real messages | shown | note |
|---|---|---|---|
| before | 2 | 2 | |
| immediately after pressing send | **2** | **3** | the third is marked "sending…" |
| after the action completes | 3 | 3 | no marker |

And with the failure checkbox ticked:

| moment | real | shown |
|---|---|---|
| immediately after send | 3 | **4** |
| after the action fails | 3 | **3** |

The optimistic entry simply **disappears**. There is no rollback code because there was nothing to
roll back: the optimistic value never entered the real state. That is the design — React's docs put
it as *"optimistic state only renders while an Action is in progress"*, and the two converge in one
render when the Action ends:

> There is no extra render to "clear" the optimistic state — the optimistic and real state converge
> in the same render when the Transition completes.

So your error handling is unchanged from LESSON 76: the Action reports the failure, you show the
message. The disappearing row is free.

### The rule

> The `set` function must be called inside an Action.

Inside a `<form action>` function you are already in one, so `addOptimistic` just works. Outside,
wrap it:

```jsx
startTransition(async () => {
  addOptimistic(text);
  await save(text);
});
```

Call it in a plain click handler with no Transition and React warns, and the optimistic state
renders only briefly.

### When it is right, and when it is a lie

Optimistic UI is a **bet that the action will succeed**. Take the bet when success is
overwhelmingly likely and a failure is cheap to explain: sending a chat message, liking a post,
ticking a to-do, reordering a list.

Do not take it when the failure is expensive or confusing: a payment, a booking that can sell out,
anything where the user will act on seeing it succeed. Showing a seat as booked and taking it away
two seconds later is worse than a spinner — the user has already told someone they got the seat.

Two practical rules if you do take it:

- **Mark it as in flight** — the "sending…" in the experiment. The user should be able to tell
  what is confirmed.
- **Make failure loud.** The row vanishing is subtle; the message explaining why must not be.

### Key Notes

- `useOptimistic(realValue, reducer)` returns the value to render plus a setter to call in an
  Action.
- The optimistic value exists only while the Action runs; when it ends, real state takes over in
  the same render.
- A failure needs no rollback — the optimistic entry never existed in real state. Say so loudly
  though.
- Bet only when success is very likely and failure is cheap. Never on money or scarce resources.

### Example

**In the playground.** Point `src/App.jsx` at `./experiments/36-optimistic.jsx`.

Send a message and watch the counters: "real" stays at 2 while "shown" is already 3. Then tick
"make sending fail" and send another — the entry appears, then vanishes, and the real count never
moved.

### Exercise

**In the playground**, in `36-optimistic.jsx`.

1. Move `addOptimistic(text)` out of the action into a plain `onClick` handler on the button. What
   does React warn, and what do you see on screen? Put it back and connect it to the rule above.
2. The optimistic row uses `id: "optimistic"`. Send two messages quickly and describe what happens
   to the keys. Fix it so several in-flight messages can coexist.
3. On failure, keep the failed message visible with a "retry" button instead of letting it vanish.
   Where must that message live — optimistic state or real state — and why does that answer follow
   from the first Key Note?
4. Reduce the mock delay to 50 ms. Is the optimistic update still worth having? Say what you would
   measure to decide.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**Runnable — plain JS.** Deciding when to bet, as a rule you can apply rather than a feeling.

Write `l83ShouldBeOptimistic(action)` taking `{ name, successRate, failureCost, reversible }` and
returning a decision plus a reason. Run it over: sending a chat message, liking a post, booking the
last seat on a flight, paying an invoice, renaming a file, and deleting an account.

Then, in comments:

1. Two of those have a high success rate and should still not be optimistic. Which, and what does
   your function use to tell them apart from the others?
2. Deleting an account is irreversible but its *failure* is cheap. Does optimistic UI apply? Say
   what the user should see instead.
3. Your function takes `successRate` as a number. Where would that number actually come from in a
   real project, and what should you do before you have one?

In [ ]:
// Your code here

> **Topic 25 complete — LESSON 78 to 83.** Repeated and dependent fields, validation as data
> shown at the right moment, and the four pieces of React's Actions: `<form action>`,
> `useActionState`, `useFormStatus` and `useOptimistic`.
>
> Read them as one thing rather than four APIs. A submit runs as a Transition; its result becomes
> state; any component inside the form can see it is pending; and the UI can run ahead of it while
> it does. Mini-project 4 uses the first three, and the fourth is one of its bonuses.